In [ ]:
import base64, os, pathlib, subprocess
from google.colab import userdata

REPO_URL = "https://github.com/devlucascfarias/logos-3.git"
REPO_BRANCH = "main"
WORKDIR = "/content/logos-3"
repo = pathlib.Path(WORKDIR)
token = userdata.get("GH_TOKEN")
git = ["git"]
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]
if (repo / ".git").exists():
    subprocess.run(git + ["-C", WORKDIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
elif repo.exists():
    raise RuntimeError(f"{WORKDIR} existe, mas não é um repositório Git")
else:
    subprocess.run(git + ["clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, WORKDIR], check=True)
token = auth = None
os.chdir(WORKDIR)
print(f"Repositório sincronizado em {os.getcwd()}")

# Treinamento corretivo `corrective_v1` em NVIDIA L4

Esta execução parte do campeão `pilot_500k_step7`, materializa um corpus assinado de 320 mil tokens e mantém o campeão intacto até todos os gates serem aprovados.

In [ ]:
STAGE = "corrective_v1"
DATA_STAGE = "corrective_v1"
RUN_DATA_PREPARATION = True
RUN_TRAINING = True
FRESH_RUN = True  # primeira execução usa resume=none; reexecute a célula de treino para auto
TOKEN_BUDGET = 320_000
MAX_STEPS = None  # obrigatório: o stage executa exatamente uma época
MAX_TRAIN_SAMPLES = None
EVAL_SEED = 20260722

PRESERVED_RUN_PATH = "/content/drive/MyDrive/logos-3/runs/pilot_500k_step7"
LOCAL_PILOT_PATH = "/content/pilot_500k_step7"
SOURCE_ADAPTER_PATH = f"{LOCAL_PILOT_PATH}/adapter"
REFERENCE_ADAPTER_PATH = SOURCE_ADAPTER_PATH
CANDIDATE_DIR = "data/interim/corrective_v1"
CANDIDATES_JSONL = f"{CANDIDATE_DIR}/candidates.jsonl"
CANDIDATE_MANIFEST = f"{CANDIDATE_DIR}/candidate_manifest.json"
EVAL_OUTPUT_DIR = "outputs/evaluations/corrective_v1"
print({"stage": STAGE, "token_budget": TOKEN_BUDGET, "fresh": FRESH_RUN})

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN não definido; OpenCodeInstruct precisa estar acessível anonimamente.")
hf_token = None
subprocess.run([sys.executable, "scripts/environment_check.py"], check=True)

## 1. Copiar e validar o campeão

O Drive é desmontado antes de qualquer código de dataset ser executado. O builder validará novamente os hashes do replay contra o manifesto copiado.

In [ ]:
import hashlib, json, pathlib, shutil
from google.colab import drive

def sha256(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()

if not pathlib.Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
source_run = pathlib.Path(PRESERVED_RUN_PATH)
local_run = pathlib.Path(LOCAL_PILOT_PATH)
required = [
    source_run / "adapter/adapter_config.json",
    source_run / "adapter/adapter_model.safetensors",
    source_run / "adapter/run_manifest.json",
    source_run / "data/train.jsonl",
    source_run / "data/validation.jsonl",
    source_run / "data/dataset_report.json",
]
missing = [str(path) for path in required if not path.exists() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError("Backup campeão incompleto: " + ", ".join(missing))
if local_run.exists():
    shutil.rmtree(local_run)
shutil.copytree(source_run, local_run)
for source_path in required:
    relative = source_path.relative_to(source_run)
    if sha256(source_path) != sha256(local_run / relative):
        raise RuntimeError(f"Falha de integridade na cópia: {relative}")
manifest = json.loads((local_run / "adapter/run_manifest.json").read_text(encoding="utf-8"))
for name, key in (("train.jsonl", "train_sha256"), ("validation.jsonl", "validation_sha256"), ("dataset_report.json", "dataset_report_sha256")):
    expected = manifest.get("data", {}).get(key)
    if not expected or sha256(local_run / "data" / name) != expected:
        raise RuntimeError(f"Hash do replay divergente ou ausente: {name}")
drive.flush_and_unmount()
print(f"Campeão e replay validados localmente em {local_run}; Drive desmontado.")

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 2. Gerar, verificar e preparar o corpus

Cada amostra nova é validada em processo isolado. Não use `--skip-execution`: esse modo produz um manifesto inelegível para treino.

In [ ]:
import json, math, pathlib, subprocess, sys
if RUN_DATA_PREPARATION:
    subprocess.run([
        sys.executable, "-u", "scripts/build_corrective_corpus.py",
        "--pilot-run-path", LOCAL_PILOT_PATH,
        "--output-dir", CANDIDATE_DIR,
        "--seed", "42",
    ], check=True)
    subprocess.run([
        sys.executable, "-u", "scripts/prepare_data.py",
        "--stage", DATA_STAGE,
        "--token-budget", str(TOKEN_BUDGET),
        "--candidates-jsonl", CANDIDATES_JSONL,
        "--candidate-manifest", CANDIDATE_MANIFEST,
    ], check=True)
report_path = pathlib.Path(f"data/processed/{DATA_STAGE}/dataset_report.json")
report = json.loads(report_path.read_text(encoding="utf-8"))
mix = report["mix"]
if mix["selected_tokens"] < 304_000:
    raise RuntimeError(f"Corpus abaixo do gate: {mix['selected_tokens']:,} tokens")
if mix["max_category_deviation"] > 0.05:
    raise RuntimeError(f"Quota fora do gate: {mix['category_deviation']}")
train_tokens = int(report["split"]["train_tokens"])
estimated_steps = math.ceil(math.ceil(train_tokens / 2048) / 8)
if not 15 <= estimated_steps <= 25:
    raise RuntimeError(f"Estimativa fora de 15–25 passos: {estimated_steps}")
print(json.dumps({"mix": mix, "split": report["split"], "optimizer_steps_estimate": estimated_steps}, ensure_ascii=False, indent=2))

## 3. Treinar com barra, ETA e log

Na primeira execução o resume é `none`. Se o runtime interromper, reexecute esta célula: ela usará `auto`. `MAX_STEPS` permanece `None`.

In [ ]:
import codecs, datetime, os, pathlib, shutil, subprocess, sys
source_adapter = pathlib.Path(SOURCE_ADAPTER_PATH)
for name in ("adapter_config.json", "adapter_model.safetensors", "run_manifest.json"):
    if not (source_adapter / name).exists():
        raise FileNotFoundError(f"Adapter inicial incompleto: {source_adapter / name}")
if MAX_STEPS is not None or MAX_TRAIN_SAMPLES is not None:
    raise ValueError("corrective_v1 proíbe MAX_STEPS e MAX_TRAIN_SAMPLES")
if RUN_TRAINING:
    fresh_run = FRESH_RUN
    if fresh_run:
        timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archive_root = pathlib.Path("outputs/archive") / f"{STAGE}-{timestamp}"
        for label, source in (("checkpoints", pathlib.Path(f"outputs/checkpoints/{STAGE}")), ("adapter", pathlib.Path(f"outputs/adapters/{STAGE}"))):
            if source.exists():
                archive_root.mkdir(parents=True, exist_ok=True)
                shutil.move(str(source), str(archive_root / label))
        FRESH_RUN = False
    command = [sys.executable, "-u", "scripts/train_sft.py", "--stage", STAGE, "--data-stage", DATA_STAGE, "--adapter-path", SOURCE_ADAPTER_PATH, "--resume-from-checkpoint", "none" if fresh_run else "auto"]
    print("Iniciando treino com barra de progresso e ETA...", flush=True)
    log_path = pathlib.Path(f"outputs/logs/{STAGE}_train.log")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    process = subprocess.Popen(command, cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    assert process.stdout is not None
    with log_path.open("wb") as log_file:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            log_file.write(chunk); log_file.flush()
            sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
        sys.stdout.write(decoder.decode(b"", final=True)); sys.stdout.flush()
    return_code = process.wait()
    print(f"\nLog salvo em: {log_path}")
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
else:
    print("RUN_TRAINING=False: corpus preparado; treino não iniciado.")

## 4. Ranqueamento no `dev_v1`

Todos os checkpoints são gerados em modo direto, greedy e com 512 tokens. Os testes ocultos são executados somente pelo avaliador.

In [ ]:
import json, pathlib, subprocess, sys
eval_root = pathlib.Path(EVAL_OUTPUT_DIR)
eval_root.mkdir(parents=True, exist_ok=True)
checkpoints = sorted(pathlib.Path(f"outputs/checkpoints/{STAGE}").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
if not checkpoints:
    raise FileNotFoundError("Nenhum checkpoint corrective_v1 encontrado.")
dev_predictions, loss_args = [], []
for checkpoint in checkpoints:
    prediction = eval_root / f"{checkpoint.name}-dev.jsonl"
    subprocess.run([sys.executable, "-u", "scripts/generate_evaluation.py", "--tasks", f"{CANDIDATE_DIR}/dev_v1.jsonl", "--adapter", str(checkpoint), "--output", str(prediction), "--decoding", "deterministic", "--seed", str(EVAL_SEED)], check=True)
    dev_predictions.append(str(prediction))
    state_path = checkpoint / "trainer_state.json"
    if state_path.exists():
        history = json.loads(state_path.read_text(encoding="utf-8")).get("log_history", [])
        losses = [item["eval_loss"] for item in history if "eval_loss" in item]
        if losses:
            loss_args += ["--eval-loss", f"{checkpoint}={losses[-1]}"]
dev_report_path = eval_root / "dev_report.json"
subprocess.run([sys.executable, "scripts/evaluate_contracts.py", "--tasks", f"{CANDIDATE_DIR}/dev_v1.jsonl", "--predictions", *dev_predictions, "--output", str(dev_report_path), *loss_args], check=True)
dev_report = json.loads(dev_report_path.read_text(encoding="utf-8"))
TOP_THREE = dev_report["top_three"]
WINNER_ADAPTER_PATH = TOP_THREE[0]
print({"top_three": TOP_THREE, "winner": WINNER_ADAPTER_PATH})

## 5. Hidden: três finalistas, campeão e base

A suíte `corrective_hidden_v1` tem 33 tarefas e decide o gate funcional.

In [ ]:
from qwen_sft.config import load_config
hidden_tasks = f"{CANDIDATE_DIR}/corrective_hidden_v1.jsonl"
hidden_predictions = []
for label, adapter in [(pathlib.Path(path).name, path) for path in TOP_THREE] + [("champion", REFERENCE_ADAPTER_PATH), ("base", None)]:
    prediction = eval_root / f"{label}-hidden.jsonl"
    command = [sys.executable, "-u", "scripts/generate_evaluation.py", "--tasks", hidden_tasks, "--output", str(prediction), "--decoding", "deterministic", "--seed", str(EVAL_SEED)]
    if adapter:
        command += ["--adapter", adapter]
    subprocess.run(command, check=True)
    hidden_predictions.append(str(prediction))
hidden_report_path = eval_root / "hidden_report.json"
subprocess.run([sys.executable, "scripts/evaluate_contracts.py", "--tasks", hidden_tasks, "--predictions", *hidden_predictions, "--output", str(hidden_report_path)], check=True)
hidden_report = json.loads(hidden_report_path.read_text(encoding="utf-8"))
BASE_IDENTITY = load_config("configs/recipe.yaml")["model_name"]
print(json.dumps(hidden_report["ranking"], ensure_ascii=False, indent=2))

## 6. Regressão cega dos 13 prompts

Avalie `comparison.md` e preencha `ratings.json` antes de abrir `mapping.json` ou executar o gate final.

In [ ]:
regression_dir = eval_root / "regression"
subprocess.run([sys.executable, "-u", "scripts/compare_adapter.py", "--stage", STAGE, "--adapter-path", WINNER_ADAPTER_PATH, "--reference-adapter-path", REFERENCE_ADAPTER_PATH, "--output-dir", str(regression_dir), "--max-new-tokens", "512", "--seed", str(EVAL_SEED), "--no-thinking"], check=True)
from IPython.display import Markdown, display
display(Markdown((regression_dir / "comparison.md").read_text(encoding="utf-8")))
print("Preencha:", regression_dir / "ratings.json")
print("Só depois abra:", regression_dir / "mapping.json")

## 7. Gate final de promoção

Depois de preencher as notas, esta célula combina hidden e regressão. Qualquer gate reprovado mantém `pilot_500k_step7` como campeão.

In [ ]:
ratings_path = regression_dir / "ratings.json"
ratings = json.loads(ratings_path.read_text(encoding="utf-8"))
if any(score is None for row in ratings for score in row["ratings"].values() for score in score.values()):
    raise RuntimeError("Preencha todas as notas de ratings.json antes do gate.")
gate_path = eval_root / "promotion_gate.json"
subprocess.run([sys.executable, "scripts/evaluate_contracts.py", "--tasks", hidden_tasks, "--predictions", *hidden_predictions, "--output", str(hidden_report_path), "--promotion-output", str(gate_path), "--ratings", str(ratings_path), "--mapping", str(regression_dir / "mapping.json"), "--candidate", WINNER_ADAPTER_PATH, "--champion", REFERENCE_ADAPTER_PATH, "--base", BASE_IDENTITY, "--candidate-rating-identity", "adapter", "--base-rating-identity", "base"], check=True)
gate = json.loads(gate_path.read_text(encoding="utf-8"))
print(json.dumps(gate, ensure_ascii=False, indent=2))

## 8. Preservar a execução

O backup inclui corpus, manifestos, logs, adapter final, avaliações e os três melhores checkpoints. Isso não altera o diretório do campeão anterior.

In [ ]:
from google.colab import drive
if not pathlib.Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
backup_root = pathlib.Path("/content/drive/MyDrive/logos-3/runs/corrective_v1")
backup_root.mkdir(parents=True, exist_ok=True)
for source, destination in [(pathlib.Path(CANDIDATE_DIR), backup_root / "candidates"), (pathlib.Path(f"data/processed/{DATA_STAGE}"), backup_root / "data"), (pathlib.Path(f"outputs/adapters/{STAGE}"), backup_root / "adapter"), (pathlib.Path(EVAL_OUTPUT_DIR), backup_root / "evaluations")]:
    if source.exists():
        shutil.copytree(source, destination, dirs_exist_ok=True)
logs_backup = backup_root / "logs"
logs_backup.mkdir(parents=True, exist_ok=True)
for log_path in pathlib.Path("outputs/logs").glob(f"{STAGE}_*.log"):
    shutil.copy2(log_path, logs_backup / log_path.name)
for checkpoint in TOP_THREE:
    source = pathlib.Path(checkpoint)
    shutil.copytree(source, backup_root / "checkpoints" / source.name, dirs_exist_ok=True)
print(f"Execução preservada em: {backup_root}")